### https://www.kaggle.com/competitions/drawing-with-llms

In [4]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator


This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


In [6]:
import mlflow
import os
os.environ['MLFLOW_TRACKING_URI'] = './mlruns'
import gc
import pandas as pd
from svg_processor import SVGSanitizer, SVGProcessor, svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator
from vllm import LLM, SamplingParams
import  re
from concurrent.futures import ThreadPoolExecutor

from tqdm import tqdm
tqdm.pandas()

class Model:
    def __init__(self):
        # MLflow experiment tracking setup
        self.experiment_name = "svg_score_test_76"
        mlflow.set_experiment(self.experiment_name)

        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        self.model = LLM(
            model=self.model_path,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.85
        )

        # Default generation parameters (can be overridden later in run_experiment)
        self.temperature = 0.5
        self.top_k = 40
        self.top_p = 0.95
        self.max_tokens = 1024

        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model
        gc.collect()

    def log_params_and_metrics(self, model_name, temperature, top_k, top_p, max_tokens, sl_score):
        """ Log parameters and metrics to MLflow """
        mlflow.log_param("model", model_name)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("top_k", top_k)
        mlflow.log_param("top_p", top_p)
        mlflow.log_param("max_tokens", max_tokens)
        mlflow.log_metric("siglip_score", sl_score)
        
    def get_response(self, description, temperature, top_k, top_p, max_tokens):
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """

        formatted_input = alpaca_prompt.format(description)
        sampling_params = SamplingParams(temperature=temperature, top_k=top_k, top_p=top_p, max_tokens=max_tokens)
        outputs = self.model.generate([formatted_input], sampling_params)

        # suitable for batch inputs as well        
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
        return generated_text

    def predict(self, description: str, temperature, top_k, top_p, max_tokens) -> str:
        output_decoded = self.get_response(description, temperature, top_k, top_p, max_tokens)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)

    def run_experiment(self, df, model_name, temperature=None, top_k=None, top_p=None, max_tokens=None):
        """ Log experiment and track prediction & evaluation """
        # Use passed parameters, or default to the instance's values
        temperature = temperature or self.temperature
        top_k = top_k or self.top_k
        top_p = top_p or self.top_p
        max_tokens = max_tokens or self.max_tokens
        
        with mlflow.start_run():
            # Generate SVG code
            df['svg'] = df['description'].apply(lambda x: self.predict(x, temperature, top_k, top_p, max_tokens))

            # Generate sl score
            df['sl_score'] = df.apply(lambda row: SVGMetricEvaluator().svg_metric(row['description'], row['svg']), axis=1)

            sl_score = df['sl_score'].mean()
            
            # Log parameters and metrics
            self.log_params_and_metrics(model_name, temperature, top_k, top_p, max_tokens, sl_score)
            
            return df, sl_score
    



INFO 04-09 23:29:55 [__init__.py:239] Automatically detected platform cuda.


In [7]:
#model instance 
model = Model()

2025/04/09 23:29:55 INFO mlflow.tracking.fluent: Experiment with name 'svg_score_test_76' does not exist. Creating a new experiment.


WARNING 04-09 23:29:55 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-09 23:30:00 [config.py:585] This model supports multiple tasks: {'classify', 'generate', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
INFO 04-09 23:30:00 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-09 23:30:01 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_b

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-09 23:30:08 [loader.py:447] Loading weights took 5.45 seconds
INFO 04-09 23:30:08 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 5.665331 seconds
INFO 04-09 23:30:15 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/84ad9dae08/rank_0_0 for vLLM's torch.compile
INFO 04-09 23:30:15 [backends.py:425] Dynamo bytecode transform time: 6.35 s
INFO 04-09 23:30:15 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-09 23:30:19 [monitor.py:33] torch.compile takes 6.35 s in total
INFO 04-09 23:30:20 [kv_cache_utils.py:566] GPU KV cache size: 18,736 tokens
INFO 04-09 23:30:20 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 18.30x
INFO 04-09 23:30:36 [gpu_model_runner.py:1534] Graph capturing finished in 16 secs, took 0.44 GiB
INFO 04-09 23:30:36 [core.py:151] init engine (profile, create kv cache, warmup model) took 27.70 seconds


In [8]:
#load df & score
#df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df=df[['description','svg']]


In [9]:
temperature_list = [0.5,0.6,0.7,0.8,0.9]
top_k_list       = [60,70,80]
top_p_list       = [0.9,0.95,0.99]
max_tokens_list  = [1024]

print('No of experiments:',len(temperature_list)*len(top_k_list)*len(top_p_list)*len(max_tokens_list) )
print('Required GPU time (Hrs):',len(temperature_list)*len(top_k_list)*len(top_p_list)*len(max_tokens_list)*7/60 )

No of experiments: 45
Required GPU time (Hrs): 5.25


In [10]:
import time

for max_tokens in max_tokens_list:
    for top_p in top_p_list:
        for top_k in top_k_list:
            for temperature in temperature_list:

                model_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
                
                current_time = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())

                df_, sl_score = model.run_experiment(df, model_name, temperature=temperature, top_k=top_k, top_p=top_p,\
                                                     max_tokens=max_tokens)
                
                df_.to_csv(f'./io_files/{model_name}_{temperature}_{top_k}_{top_p}_{max_tokens}_\
                                                    {current_time}.csv')
                
                print('sl_score',sl_score)

Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.43s/it, est. speed input: 7.35 t
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.20s/it, est. speed input: 14.53 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.55s/it, est. speed input: 23.53 
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.37s/it, est. speed input: 9.58 t
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.86s/it, est. speed input: 9.04 t
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.86s/it, est. speed input: 12.97 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.11s/it, est. speed input: 29.33 
Processed prompts: 100%|█| 1/1 [00:01<00:00,  1.93s/it, est. speed input: 32.10 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.61s/it, est. speed input: 13.02 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.96s/it, est. speed input: 21.31 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.06s/it, est. speed input: 19.58 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.56s/it, est. speed input: 11.16 
Processed prompts: 100%|█| 1

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

Processed prompts: 100%|█| 1/1 [00:11<00:00, 11.38s/it, est. speed input: 5.45 t
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 45, column 14 (<string>, line 45). Returning default SVG.
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.64s/it, est. speed input: 16.75 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.37s/it, est. speed input: 17.80 
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.20s/it, est. speed input: 7.44 t
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.03s/it, est. speed input: 15.39 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.41s/it, est. speed input: 11.65 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.19s/it, est. speed input: 19.44 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.91s/it, est. speed input: 21.31 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.62s/it, est. speed input: 13.00 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.28s/it, est. speed input: 19.19 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.80s

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.07s/it, est. speed input: 7.68 t
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.04s/it, est. speed input: 20.08 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.62s/it, est. speed input: 22.87 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.22s/it, est. speed input: 11.69 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.77s/it, est. speed input: 10.75 
Processed prompts: 100%|█| 1/1 [00:09<00:00,  9.94s/it, est. speed input: 6.34 t
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.02s/it, est. speed input: 20.52 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.91s/it, est. speed input: 21.29 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.03s/it, est. speed input: 14.91 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.33s/it, est. speed input: 18.93 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.23s/it, est. speed input: 18.55 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.90s/it, est. speed input: 10.51 
Processed prompts: 100%|█| 1

...

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.52s/it, est. speed input: 11.24 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.58s/it, est. speed input: 17.03 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.10s/it, est. speed input: 19.33 
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.58s/it, est. speed input: 8.05 t
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.25s/it, est. speed input: 14.61 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.81s/it, est. speed input: 13.10 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.41s/it, est. speed input: 18.20 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.48s/it, est. speed input: 17.82 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.61s/it, est. speed input: 13.03 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.60s/it, est. speed input: 24.26 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.01s/it, est. speed input: 19.95 
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.43s/it, est. speed input: 7.35 t
Processed prompts: 100%|█| 1

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.71s/it, est. speed input: 10.86 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.04s/it, est. speed input: 29.96 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.90s/it, est. speed input: 20.67 
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.65s/it, est. speed input: 10.80 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.94s/it, est. speed input: 12.54 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.74s/it, est. speed input: 23.03 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.50s/it, est. speed input: 17.71 
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.06s/it, est. speed input: 30.13 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.62s/it, est. speed input: 13.00 
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.31s/it, est. speed input: 19.02 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.22s/it, est. speed input: 14.22 
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.26s/it, est. speed input: 14.54 
Processed prompts: 100%|█| 1

Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device

In [11]:
#model.close_model()